Here's your plan:

- input → tokens
- token embedding + positional embedding (add them)
- one-head attention ✓ (done)
- multi-head attention — heads in parallel, concat, + an output projection back to n_embd
- feedforward (Linear → ReLU → Linear, 4× wide) ← was missing
- assemble a Block = LN→MHA→+residual, LN→FFN→+residual
- stack N blocks (depth)
- final LayerNorm + lm_head (Linear n_embd → vocab_size to get logits) ← easy to forget
- forward → cross-entropy loss (reshape logits to (B*T, vocab), targets to (B*T,))
- backward + optimizer step (zero_grad → backward → step, AdamW) — this is your "training loop"
- sampling / generate

In [11]:
import torch 
import torch.nn.functional as F
import torch.nn as nn 

In [12]:
with open('./input.txt','r') as f:
    x = f.read()

In [65]:
T = 4
B = 10
C = 2
head_size = 5

In [14]:
inp = torch.randint(1,len(x)-T,(B,))
inp = [x[i.item():i.item()+T] for i in inp]
stoi = {}
itos = {}
for i,s in enumerate(sorted(list(set(x)))):
    stoi[s] = i 
    itos[i] = s 


inp2 = []
for i in inp:
    inp2.append([stoi[j] for j in i])

inp2 = torch.tensor(inp2)
inp2

tensor([[39, 49, 43,  1],
        [ 1, 39, 52, 42],
        [56, 43,  6,  1],
        [56, 47, 45, 46],
        [59, 57,  1, 50],
        [61, 43,  1, 50],
        [43,  1, 39,  1],
        [53, 39, 56,  0],
        [43,  1, 41, 53],
        [56, 42,  1, 45]])

cleaner implementaion 

In [73]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embd = nn.Embedding(len(stoi),C)
        self.posi = nn.Embedding(T,C)
        self.query = nn.Linear(C,head_size,bias=False)
        self.key = nn.Linear(C,head_size,bias=False)
        self.value = nn.Linear(C,head_size,bias=False)

    def nn_embed(self,x):
        embedings = self.embd(x)
        embedings = embedings+self.posi(torch.arange(T))
        return embedings

    def attention(self,x):
        query = self.query(x)
        key = self.key(x)
        value = self.value(x)

        qk = query@key.transpose(-2,-1) * head_size**-0.5
        tril = torch.tril(torch.ones(T,T))
        we = qk.masked_fill(tril == 0, float('-inf'))
        we = torch.softmax(we,dim=-1)

        return we@value

        

In [74]:
gpt = GPT()

In [75]:
emb = gpt.nn_embed(inp2)
att = gpt.attention(emb)

In [76]:
att

tensor([[[-0.5924, -0.6954,  0.4904,  0.1795, -0.4897],
         [-0.7101, -0.6794,  0.5170,  0.3416, -0.3779],
         [-0.7879, -0.5448,  0.4777,  0.5501, -0.1360],
         [-0.8274, -0.1393,  0.3029,  0.9323,  0.4438]],

        [[-0.4677, -0.6218,  0.4206,  0.0821, -0.4853],
         [-0.4770, -0.3264,  0.2876,  0.3359, -0.0777],
         [-0.2842, -0.3330,  0.2350,  0.0867, -0.2340],
         [-0.1849, -0.3689,  0.2228, -0.0683, -0.3587]],

        [[-1.1822, -1.5125,  1.0359,  0.2561, -1.1463],
         [-1.2496, -1.2307,  0.9260,  0.5722, -0.7127],
         [-1.4829,  0.0292,  0.4148,  1.8993,  1.1735],
         [-1.0394, -0.2507,  0.4152,  1.1091,  0.4550]],

        [[-1.1822, -1.5125,  1.0359,  0.2561, -1.1463],
         [-0.8056, -0.8261,  0.6120,  0.3422, -0.5037],
         [-0.9001, -1.1217,  0.7750,  0.2194, -0.8323],
         [-1.0755, -1.3354,  0.9238,  0.2663, -0.9877]],

        [[-0.3910, -0.8431,  0.5001, -0.1960, -0.8438],
         [-0.7064, -0.5047,  0.4357,  0.